# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records from the Croissant schema.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs using Croissant metadata.

In [ ]:
# List all available record sets and their IDs
print("Available Record Sets:")
record_sets = dataset.record_sets
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}")

# For each RecordSet, list its fields and columns by @id
for rs in record_sets:
    print(f"\nRecordSet: {rs['@id']}  Name: {rs.get('name', '')}")
    if 'field' in rs:
        fields = rs['field']
        print("  Fields:")
        for f in fields:
            # field can be dict or just id
            if isinstance(f, dict):
                print(f"    - {f.get('@id', f)}")
            else:
                print(f"    - {f}")
    if 'column' in rs:
        columns = rs['column']
        print("  Columns:")
        for c in columns:
            if isinstance(c, dict):
                print(f"    - {c.get('@id', c)}")
            else:
                print(f"    - {c}")

## 3. Data Extraction
Load data from specific record sets into Pandas DataFrames. Use the record set and field `@id`s identified above.

In [ ]:
# Prepare: choose record sets by their @id for extraction (use the actual IDs printed above). In this example, we use placeholder IDs.
# Please refer to the Data Overview output for the exact @id values in your environment.

# Example: Suppose there is one main record set: 'cr:ProteinAbundanceRecordSet'
main_record_set_id = 'cr:ProteinAbundanceRecordSet'  # <-- UPDATE if your dataset shows a different @id
record_sets_ids = [main_record_set_id]

dataframes = {}
for rs_id in record_sets_ids:
    print(f"Loading records for RecordSet: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Fields/columns: {df.columns.tolist()}")
        display(df.head())
    else:
        print("No records found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering, normalizing, grouping, and summarizing numeric fields. All field access is by Croissant `@id`.

In [ ]:
# Choose a numeric field for analysis. Example: suppose '@id': 'cr:abundance'
numeric_field_id = 'cr:abundance'  # <-- UPDATE this according to the Data Overview output

# Ensure the record set exists
df = dataframes.get(main_record_set_id)
if df is not None and numeric_field_id in df:
    # Safely convert field to numeric
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    normalized_col = f"{numeric_field_id}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id}:")
    display(filtered_df[[numeric_field_id, normalized_col]].head())

    # Group by a categorical field
    group_field_id = 'cr:sample'  # <-- UPDATE this to a categorical field's @id if available
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"Mean {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
    else:
        print(f"Field {group_field_id} not found for grouping.")
else:
    print(f"DataFrame or field {numeric_field_id} not found. Please update 'numeric_field_id' as needed.")

## 5. Visualization
Visualize the distribution of numeric attributes or relationships in your dataset.

In [ ]:
import matplotlib.pyplot as plt

if df is not None and numeric_field_id in df:
    # Histogram of the numeric field
    df[numeric_field_id].hist(bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # Boxplot grouped by a categorical variable (if present)
    if group_field_id in df.columns:
        df.boxplot(column=numeric_field_id, by=group_field_id, rot=90, grid=False)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.suptitle("")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No valid DataFrame or field found for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration. Adjust the field `@id` values in the code above to match those reported in your Data Overview section (Step 2), as necessary. This notebook has demonstrated end-to-end data loading, overview, and analysis using the FAIR² dataset and the `mlcroissant` library.